# Workshop #2 — Silver Layer Parquet Data Exploration
**Course:** Programación para Análisis de Datos (Data Analysis Programming) — Semester 2026-II  
**Team 1:** Natural Hazard Monitoring Systems — Earthquake Activity Monitoring  
**Authors:** Juan Córdoba (20211020097), Angelo Ibáñez (20212020007), Mauricio Sánchez (20212020045)  

---

## Objective
This notebook demonstrates the loading, schema validation, and statistical profiling of curated earthquake events persisted in the **Silver Data Lake layer** (`datalake_silver/`) formatted as Apache Parquet.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

SILVER_DIR = Path("../datalake_silver")
parquet_files = sorted(SILVER_DIR.glob("*.parquet"))
print(f"Found {len(parquet_files)} Silver Parquet file(s):\n")
for pf in parquet_files:
    size_kb = pf.stat().st_size / 1024.0
    print(f"  - {pf.name} ({size_kb:.2f} KB)")

latest_file = parquet_files[-1]
print(f"\nLoading latest curated file: {latest_file.name}")
df = pd.read_parquet(latest_file)
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")

Found 3 Silver Parquet file(s):

  - earthquakes_silver_20261117_013502.parquet (24.81 KB)
  - earthquakes_silver_20261117_013535.parquet (24.81 KB)
  - earthquakes_silver_20261117_013549.parquet (24.81 KB)

Loading latest curated file: earthquakes_silver_20261117_013549.parquet
Dataset shape: 245 rows x 13 columns


## 1. Schema & Data Types Inspection
Confirms proper typing for coordinates (float), timestamps (datetime), and categorical indicators.

In [2]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 245 entries, 0 to 244
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   event_id        245 non-null    str                
 1   raw_time        245 non-null    int64              
 2   magnitude       245 non-null    float64            
 3   mag_type        245 non-null    str                
 4   place           245 non-null    str                
 5   longitude       245 non-null    float64            
 6   latitude        245 non-null    float64            
 7   depth_km        245 non-null    float64            
 8   tsunami         245 non-null    bool               
 9   source_file     245 non-null    str                
 10  event_time      245 non-null    datetime64[ns, UTC]
 11  magnitude_type  245 non-null    str                
 12  energy_joules   245 non-null    float64            
dtypes: bool(1), datetime64[ns, UTC](1), float64(5)

## 2. Null Value & Data Quality Verification
Confirms that all critical coordinates, identifiers, and magnitudes are 100% complete.

In [3]:
null_summary = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing (%)": (df.isnull().sum() / len(df)) * 100
})
print(null_summary)

                Missing Values  Missing (%)
event_id                     0          0.0
raw_time                     0          0.0
magnitude                    0          0.0
mag_type                     0          0.0
place                        0          0.0
longitude                    0          0.0
latitude                     0          0.0
depth_km                     0          0.0
tsunami                      0          0.0
source_file                  0          0.0
event_time                   0          0.0
magnitude_type               0          0.0
energy_joules                0          0.0


## 3. Descriptive Statistical Analysis
Examines magnitude ($M$), focal depth ($km$), and radiated seismic energy ($E$ in Joules).

In [4]:
numeric_cols = ["magnitude", "latitude", "longitude", "depth_km", "energy_joules"]
print(df[numeric_cols].describe().round(3))

print("\n--- Top 5 Strongest Earthquakes in Batch ---")
print(df.sort_values("magnitude", ascending=False)[["event_id", "event_time", "magnitude", "depth_km", "place"]].head())

       magnitude  latitude  longitude  depth_km  energy_joules
count    245.000   245.000    245.000   245.000   2.450000e+02
mean       2.453    37.028   -107.085    37.281   2.255896e+12
std        1.034    22.928     77.747    78.654   3.202463e+13
min        1.500   -50.374   -178.571    -0.690   1.122018e+07
25%        1.790    28.537   -151.469     6.426   3.054921e+07
50%        2.020    37.336   -121.304    11.770   6.760830e+07
75%        2.700    59.858   -112.920    30.790   7.079458e+08
max        6.600    65.174    179.175   626.486   5.011872e+14

--- Top 5 Strongest Earthquakes in Batch ---
       event_id                    event_time  magnitude  depth_km                                         place
145  us7000tymu 2026-11-16 21:11:20.515+00:00        6.6    68.000                 148 km WSW of Tual, Indonesia
73   us7000ty84 2026-11-14 09:26:13.073+00:00        5.5    10.000                   northern Mid-Atlantic Ridge
162  us7000tykh 2026-11-16 16:16:45.265+00:00   

## 4. Provenance & Source File Traceability
Demonstrates data lineage from Silver Parquet back to the generating Bronze JSON file.

In [5]:
lineage = df["source_file"].value_counts()
print("Record distribution by Bronze source file:")
print(lineage)

Record distribution by Bronze source file:
source_file
earthquakes_20261117_011500.json    128
earthquakes_20261117_013250.json    117
